# Single-modality baseline models: Elastic Net 

### Preparation

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_validate, GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.exceptions import ConvergenceWarning
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from config import HARMONIZED_DATA_PATH

In [ ]:
df = pd.read_parquet(HARMONIZED_DATA_PATH)
df.head()

,ModelID,SMILES,SequencingID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),...,Bit_1022,Bit_1023,Donor,Acceptor,Aromatic,Hydrophobe,LumpedHydrophobe,PosIonizable,NegIonizable,ZnBinder
0,ACH-000001,B(C1=CC2=CC=CC=C2S1)(O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,2.0,3.0,2.0,0.0,0.0,0.0
1,ACH-000001,B(C1=CC=CC=C1)(C2=CC=CC=C2)OCCN,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,1.0,2.0,2.0,2.0,1.0,0.0,0.0
2,ACH-000001,C#CCCCCCCCCCCCCCCCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,2.0,0.0,16.0,0.0,0.0,1.0,1.0
3,ACH-000001,C(C(=O)O)S,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,0.0,1.0,0.0,0.0,1.0,2.0
4,ACH-000001,C(C(C(=O)O)N)SCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,3.0,4.0,0.0,2.0,0.0,1.0,2.0,2.0


In [3]:
# get MFP in separate columns
if 'Bit_0' not in df.columns:
    # get MorganFP as separate columns
    fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index)
    fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
    df = pd.concat([df, fp_df], axis=1)
# get pharmacophore features as separate columns
if 'Donor' not in df.columns:
    expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist())
    df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
    df = df.fillna(0)

In [ ]:
def genomic_baseline_model(df, target):
    '''
    Function to train an Elastic Net model on genomic features and evaluate its performance.
    Parameters:
    - df: DataFrame containing the data.
    - target: The target variable for prediction.
    '''
    warnings.filterwarnings("ignore", category=ConvergenceWarning)

    # ensures that all rows from one cell line stay together in the train/test split
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['ModelID']))
    #df_train = df.iloc[train_idx].copy()
    #df_test = df.iloc[test_idx].copy()

    X_train = df.loc[df.index[train_idx], df.columns[df.columns.str.contains(r'.* \(.*\)')]].values.astype('float32')
    y_train = df.loc[df.index[train_idx], target].values.astype('float32')
    groups_train = df.loc[df.index[train_idx], 'ModelID'].values

    X_test = df.loc[df.index[test_idx], df.columns[df.columns.str.contains(r'.* \(.*\)')]].values.astype('float32')
    y_test = df.loc[df.index[test_idx], target].values.astype('float32')

    print(f"Number of unique cell lines in training set: {len(np.unique(groups_train))}")
    print(f"Number of unique cell lines in test set: {len(np.unique(df.loc[df.index[test_idx], 'ModelID'].values))}")

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNetCV(
            l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
            cv=GroupKFold(n_splits=5).split(X_train, y_train, groups=groups_train),
            random_state=42,
            max_iter=8000,
            alphas=20,
            tol=1e-3,
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train, y_train)

    train_r2_global = pipeline.score(X_train, y_train)
    fitted_model = pipeline.named_steps['model']
    best_alpha_idx = np.where(fitted_model.alphas_ == fitted_model.alpha_)[0][0]
    mean_mse_best_alpha = np.mean(fitted_model.mse_path_[best_alpha_idx])
    variance_y_train = np.var(y_train)
    train_cv_r2 = 1 - (mean_mse_best_alpha / variance_y_train)

    y_pred = pipeline.predict(X_test)
    test_r2 = pipeline.score(X_test, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print("\n" + "="*40)
    print("   ELASTIC NET CV PIPELINE ERGEBNISSE")
    print("="*40)
    print(f"Chosen Alpha:           {fitted_model.alpha_:.6f}")
    print(f"Chosen L1-Ratio:        {fitted_model.l1_ratio_:.2f}")
    print("-"*40)
    print(f"Internal CV training R²:     {train_cv_r2:.4f}")
    print(f"Test R² Score:       {test_r2:.4f}")
    print(f"Test RMSE:     {test_rmse:.4f}")
    print("="*40)

    # analyze feature importance
    final_model = pipeline.named_steps['model']
    coefs = final_model.coef_

    # Get the gene names from your original columns
    gene_names = df.loc[df.index[train_idx], df.columns[df.columns.str.contains(r'.* \(.*\)')]].columns

    # Create a summary table
    features_df = pd.DataFrame({'Gene': gene_names, 'Coefficient': coefs})
    features_df['Abs_Coef'] = features_df['Coefficient'].abs()

    # Filter for genes the model didn't set to zero
    selected_genes = features_df[features_df['Coefficient'] != 0]

    print(f"\nElastic Net selected {len(selected_genes)} genes out of 978.")
    print(f"Top 5 Positive Biomarkers (Increase {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
    print(f"\nTop 5 Negative Biomarkers (Decrease {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=True).head(5))
        

In [4]:
from sklearn.compose import ColumnTransformer

In [18]:
def chemical_baseline_model(df, target):
    '''
    Function to train an Elastic Net model on chemical features and evaluate its performance.
    Parameters:
    - df: DataFrame containing the data.
    - target: The target variable for prediction.
    '''
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    chem_cols = ('Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder', 'Bit_')

    # get MFP in separate columns
    if 'Bit_0' not in df.columns:
        # get MorganFP as separate columns
        fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index)
        fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
        df = pd.concat([df, fp_df], axis=1)
    # get pharmacophore features as separate columns
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), index=df.index)
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)
        
    # ensures that all rows from one cell line stay together in the train/test split
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['DRUG_ID']))

    X_train = df.loc[df.index[train_idx], df.columns[df.columns.str.startswith(chem_cols)]]#.values.astype('float32')
    y_train = df.loc[df.index[train_idx], target].values.astype('float32')
    groups_train = df.loc[df.index[train_idx], 'DRUG_ID'].values

    X_test = df.loc[df.index[test_idx], df.columns[df.columns.str.startswith(chem_cols)]]#.values.astype('float32')
    y_test = df.loc[df.index[test_idx], target].values.astype('float32')

    print(f"Number of unique drugs in train set: {len(np.unique(groups_train))}")
    print(f"Number of unique drugs in test set: {len(np.unique(df.loc[df.index[test_idx], 'DRUG_ID'].values))}")

    # scale pharmacophore features
    pharmacophore_cols = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    mfp_cols = [col for col in df.columns if col.startswith('Bit_')]
    preprocessor = ColumnTransformer(
        transformers=[
            ('pharmacophore', StandardScaler(), pharmacophore_cols),
            ('mfp', 'passthrough', mfp_cols)
        ],
        remainder='drop'
    )
    # Use ColumnTransformer to scale the pharmacophore features and ElasticNetCV for regression with cross-validation
    pipeline = Pipeline([
        ('scaler', preprocessor),
        ('model', ElasticNetCV(
            l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
            cv=GroupKFold(n_splits=5).split(X_train, y_train, groups=groups_train),
            random_state=42,
            max_iter=10000,
            alphas=20,
            tol=1e-3,
            n_jobs=1 # Keeping it to 1 to avoid memory issues
        ))
    ])

    print("\nFitting final model on entire training set...")
    pipeline.fit(X_train, y_train)
    # neuer versuch
    train_r2_global = pipeline.score(X_train, y_train)
    fitted_model = pipeline.named_steps['model']
    best_alpha_idx = np.where(fitted_model.alphas_ == fitted_model.alpha_)[0][0]
    mean_mse_best_alpha = np.mean(fitted_model.mse_path_[best_alpha_idx])
    variance_y_train = np.var(y_train)
    train_cv_r2 = 1 - (mean_mse_best_alpha / variance_y_train)
    
    y_pred = pipeline.predict(X_test)
    if target == 'AUC':
        y_pred = np.clip(y_pred, 0, 1)  # Clip predictions to the range [0, 1] for AUC
    test_r2 = pipeline.score(X_test, y_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # --- AUSGABE ---
    print("\n" + "="*40)
    print("   ELASTIC NET CV PIPELINE ERGEBNISSE")
    print("="*40)
    print(f"Alpha:           {fitted_model.alpha_:.6f}")
    print(f"L1-Ratio:        {fitted_model.l1_ratio_:.2f}")
    print("-"*40)
    print(f"Internal CV Training R²:     {train_cv_r2:.4f}")
    print("="*40)

    print(f"\n--- Held-Out Test Results ---")
    print(f"Test R² Score: {test_r2:.4f}")
    print(f"Test RMSE:     {test_rmse:.4f}")

    final_model = pipeline.named_steps['model']
    coefs = final_model.coef_

    # Get the molecule structure names from your original columns
    bit_names = df.columns[df.columns.str.startswith(chem_cols)]

    # Create a summary table
    features_df = pd.DataFrame({'Molecule_Structure': bit_names, 'Coefficient': coefs})
    features_df['Abs_Coef'] = features_df['Coefficient'].abs()

    # Filter for genes the model didn't set to zero
    selected_struct = features_df[features_df['Coefficient'] != 0]

    print(f"\nElastic Net selected {len(selected_struct)} molecule structures out of 1032.")
    print(f"Top 5 Positive molecule structures (Increase {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
    print(f"\nTop 5 Negative molecule structures (Decrease {target}):")
    print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

## AUC

In [6]:
genomic_baseline_model(df, "AUC")

Number of unique cell lines in training set: 572
Number of unique cell lines in test set: 143

   ELASTIC NET CV PIPELINE ERGEBNISSE
Ausgewähltes Alpha:           1.141587
Ausgewähltes L1-Ratio:        0.00
----------------------------------------
Globaler Trainings R² Score:  0.0340
Interner CV Trainings R²:     0.0135
Held-Out Test R² Score:       0.0179
Test RMSE:     0.1435

Elastic Net selected 239 genes out of 978.
Top 5 Positive Biomarkers (Increase AUC):
             Gene  Coefficient  Abs_Coef
903  PTPN1 (5770)     0.001158  0.001158
853    JUN (3725)     0.001143  0.001143
170  GNA11 (2767)     0.001039  0.001039
626   TCTA (6988)     0.000971  0.000971
284   RHEB (6009)     0.000878  0.000878

Top 5 Negative Biomarkers (Decrease AUC):
               Gene  Coefficient  Abs_Coef
148      ME2 (4200)    -0.001050  0.001050
244      CSK (1445)    -0.001022  0.001022
744     CASP3 (836)    -0.000873  0.000873
82     PRKCQ (5588)    -0.000818  0.000818
470  FBXL12 (54850)    -0.000

In [19]:
chemical_baseline_model(df, "AUC")

Number of unique drugs in train set: 190
Number of unique drugs in test set: 48

Fitting final model on entire training set...

   ELASTIC NET CV PIPELINE ERGEBNISSE
Alpha:           0.203401
L1-Ratio:        0.05
----------------------------------------
Internal CV Training R²:     -0.2991

--- Held-Out Test Results ---
Test R² Score: -0.0000
Test RMSE:     0.1320

Elastic Net selected 0 molecule structures out of 1032.
Top 5 Positive molecule structures (Increase AUC):
  Molecule_Structure  Coefficient  Abs_Coef
0              Bit_0          0.0       0.0
1              Bit_1          0.0       0.0
2              Bit_2          0.0       0.0
3              Bit_3          0.0       0.0
4              Bit_4          0.0       0.0

Top 5 Negative molecule structures (Decrease AUC):
  Molecule_Structure  Coefficient  Abs_Coef
0              Bit_0          0.0       0.0
1              Bit_1          0.0       0.0
2              Bit_2          0.0       0.0
3              Bit_3          0.

## LN_IC50

In [5]:
genomic_baseline_model(df, "LN_IC50")

Number of unique cell lines in training set: 572
Number of unique cell lines in test set: 143

   ELASTIC NET CV PIPELINE ERGEBNISSE
Ausgewähltes Alpha:           2.354932
Ausgewähltes L1-Ratio:        0.00
----------------------------------------
Globaler Trainings R² Score:  0.0771
Interner CV Trainings R²:     0.0384
Held-Out Test R² Score:       0.0460
Test RMSE:     2.7445

Elastic Net selected 878 genes out of 978.
Top 5 Positive Biomarkers (Increase LN_IC50):
               Gene  Coefficient  Abs_Coef
964  PLEKHM1 (9842)     0.021408  0.021408
53    JADE2 (23338)     0.020516  0.020516
458      BMP4 (652)     0.019443  0.019443
472   MACF1 (23499)     0.018934  0.018934
716   SQSTM1 (8878)     0.016601  0.016601

Top 5 Negative Biomarkers (Decrease LN_IC50):
               Gene  Coefficient  Abs_Coef
744     CASP3 (836)    -0.018036  0.018036
148      ME2 (4200)    -0.015626  0.015626
273  FKBP14 (55033)    -0.014896  0.014896
429    SOCS2 (8835)    -0.014231  0.014231
889   INS

In [16]:
chemical_baseline_model(df, 'LN_IC50')

Number of unique drugs in train set: 190
Number of unique drugs in test set: 48

Fitting final model on entire training set...

   ELASTIC NET CV PIPELINE ERGEBNISSE
Alpha:           0.628219
L1-Ratio:        0.05
----------------------------------------
Internal CV Training R²:     -0.0805

--- Held-Out Test Results ---
Test R² Score: 0.0334
Test RMSE:     2.5811

Elastic Net selected 188 molecule structures out of 1032.
Top 5 Positive molecule structures (Increase LN_IC50):
    Molecule_Structure  Coefficient  Abs_Coef
3                Bit_3     0.223564  0.223564
9                Bit_9     0.137905  0.137905
276            Bit_276     0.108311  0.108311
895            Bit_895     0.101091  0.101091
72              Bit_72     0.087752  0.087752

Top 5 Negative molecule structures (Decrease LN_IC50):
    Molecule_Structure  Coefficient  Abs_Coef
6                Bit_6    -0.174263  0.174263
849            Bit_849    -0.110933  0.110933
1                Bit_1    -0.102050  0.102050
822